In [2]:
from datasets import load_dataset
docs = load_dataset("thesafetystack/the-safety-stack", split="train")   # one row per document, every annotation

Generating train split: 39269 examples [00:00, 65499.93 examples/s]


In [3]:
def used(row, slug):   # the document's own work uses the technique, at least a paragraph's worth
    st = row["annotations"]["safety_techniques"]
    return any(t["slug"] == slug and t["roles"]["used"] >= 2 for t in (st and st["payload"]["techniques"]) or [])

rlhf = docs.filter(lambda r: used(r, "rlhf"))

Filter: 100%|██████████| 39269/39269 [00:03<00:00, 11996.57 examples/s]


In [4]:
rlhf

Dataset({
    features: ['release_id', 'doc_id', 'title', 'abstract', 'authors', 'published_date', 'published_precision', 'doc_type', 'is_withdrawn', 'canonical_url', 'field_sources', 'identity', 'content', 'release', 'annotations'],
    num_rows: 1861
})

In [5]:
for row in rlhf:
    print(row)

{'release_id': '2026-10-27', 'doc_id': 's00fx2c51yceh', 'title': 'The Alignment Game: A Theory of Long-Horizon Alignment Through Recursive Curation', 'abstract': None, 'authors': [{'name': 'Falahati, Ali', 'orcid': None, 'affiliation_raw': None}, {'name': 'Mohammad Mohammadi Amiri', 'orcid': '0000-0002-7343-6628', 'affiliation_raw': None}, {'name': 'Kate S. Larson', 'orcid': '0000-0002-5455-9657', 'affiliation_raw': None}, {'name': 'Lukasz Golab', 'orcid': '0000-0003-0632-7496', 'affiliation_raw': None}], 'published_date': datetime.date(2025, 11, 16), 'published_precision': 'day', 'doc_type': 'preprint', 'is_withdrawn': False, 'canonical_url': 'https://arxiv.org/abs/2511.12804', 'field_sources': [{'field': 'title', 'source_id': 's00fx2c51yceh', 'origin': 'provider'}, {'field': 'abstract', 'source_id': 's00fx2c51yceh', 'origin': 'provider'}, {'field': 'authors', 'source_id': 's00fx2c51yceh', 'origin': 'provider'}, {'field': 'published_at', 'source_id': 's00fx2c51yceh', 'origin': 'provid

## 2026 sources not yet in the Shallow Review

`find_new_sources.py` filters the dataset to 2026 and drops every document already in the Shallow Review 2026 sources (the arXiv and LessWrong scrapes in `scraping-scripts`, `facets-shared/input`, and the main pipeline's 2026 links), matching on arXiv ID, LessWrong post ID, URL or title. Rerun it with `uv run python find_new_sources.py` to refresh `output/`.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 100)

new = pd.read_csv("output/safety-stack-2026-new.csv", parse_dates=["published_date"])
matches = pd.read_csv("output/safety-stack-2026-matches.csv")
print(f"{len(new)} new, {len(matches)} already in the Shallow Review sources")

In [ ]:
# Relevant to the Shallow Review and published before the scrapes stopped (Oct 10): the scrapers missed these
missed = new[(new.sr_verdict == "relevant") & ~new.after_scrape_cutoff]
pd.crosstab(missed.domain.where(missed.domain.isin(["arxiv.org", "doi.org", "openai.com"]), "other"), missed.sr_category, margins=True)

In [ ]:
missed.sr_agenda.value_counts().head(25)

In [ ]:
# Lab blogs, system cards and other non-paper sources the scrapers don't cover
missed[~missed.doc_type.isin(["preprint", "paper", "journal", "conference"])][["published_date", "domain", "doc_type", "title", "url"]].sort_values("domain")

In [ ]:
# Browse by Shallow Review agenda
missed[missed.sr_agenda == "chain_of_thought_monitoring"][["published_date", "title", "url", "sr_quote"]]